# Day 5: Nextflow basics: channels, operators and processes

**Course:** Computational Workflows for Biomedical Data (CWBD), University of Tübingen, 2026 \
**Date:** 2026-10-02 \
**Author:** Laurens Hahn

**What:** The building blocks of the Nextflow language, one small script per topic: channel factories, operators that reshape channels, processes, meta maps from a samplesheet, and finally processes linked by channels.

**Why:** The nf-core pipelines of days 1 to 3 are nothing but processes connected by channels. Reading or writing a pipeline means knowing which operator turns one channel shape into the next.

## Nextflow concepts

### 1. Describe the concept of Workflows, Processes and Channels we deal with in Nextflow

> Describe the concept of Workflows, Processes and Channels we deal with in Nextflow

**Workflows:** A workflow is the wiring diagram. It calls processes and operators and connects them through channels, but does not run anything step by step: Nextflow reads the whole workflow as a dataflow graph and starts each task as soon as its inputs are there. Workflows can be named and nested (subworkflows); the unnamed one is the entry point that `nextflow run` executes.

**Processes:** A process is one step that runs a script (Bash by default, any language via a shebang) with declared inputs and outputs. Every item that arrives on its input channel becomes a separate task, executed in its own directory under `work/` and, if configured, inside a container or on a cluster node. Because inputs and outputs are declared, Nextflow can cache a task by hashing them and skip it on `-resume`.

**Channels:** A channel is the asynchronous queue between processes and operators. A *queue channel* (`channel.of`, `channel.fromPath`, process outputs) emits a sequence of items, each item is consumed once. A *value channel* (`channel.value`, `collect`, `first`) holds a single item that can be read any number of times. Operators such as `map`, `groupTuple` or `branch` take channels and return new ones, so the data never has to be handled with loops.

## Introduction to channels

Solutions in `channels_intro.nf`. The server runs Nextflow 26.04, whose strict parser no longer turns `--step 1` into a number, so the template's `params.step = 0` never matched any step. All scripts therefore declare the parameter with a type, `params { step: Integer = 0 }`, as the Nextflow docs recommend.

### Task 1: numbers 1 to 10

> Create a channel that enumerates the numbers from 1 to 10

`channel.of(1..10)`: a range given to `channel.of` is emitted item by item.

In [1]:
!nextflow run channels_intro.nf --step 1


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [deadly_hoover] revision: 716a17fe13

1
2
3
4
5
6
7
8
9
10


### Task 2: the alphabet

> Create a channel that gives out the entire alphabet

`channel.of('a'..'z')`: ranges work on characters as well.

In [2]:
!nextflow run channels_intro.nf --step 2


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [elated_hopper] revision: 716a17fe13

a
b
c
d
e
f
g
h
i
j
k
l
m
n
o
p
q
r
s
t
u
v
w
x
y
z


### Task 3: all files in files_dir

> Create a channel that includes all files in the "files_dir" directory

`channel.fromPath('files_dir/*')` emits one absolute path per matching file.

In [3]:
!nextflow run channels_intro.nf --step 3


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [goofy_meninsky] revision: 716a17fe13

/data/cwbd/day_05/files_dir/fastq5_2.fq
/data/cwbd/day_05/files_dir/fastq_2.fq
/data/cwbd/day_05/files_dir/SRR3_2.fq
/data/cwbd/day_05/files_dir/fastq4_1.fq
/data/cwbd/day_05/files_dir/file_3.txt
/data/cwbd/day_05/files_dir/fastq2_2.fq
/data/cwbd/day_05/files_dir/SRR1_2.fq
/data/cwbd/day_05/files_dir/file_4.txt
/data/cwbd/day_05/files_dir/fastq2_1.fq
/data/cwbd/day_05/files_dir/SRR4_2.fq
/data/cwbd/day_05/files_dir/fastq5_1.fq
/data/cwbd/day_05/files_dir/fastq1_1.fq
/data/cwbd/day_05/files_dir/SRR5_2.fq
/data/cwbd/day_05/files_dir/file_1.txt
/data/cwbd/day_05/files_dir/SRR.fq
/data/cwbd/day_05/files_dir/SRR3_1.fq
/data/cwbd/day_05/files_dir/fastq3_2.fq
/data/cwbd/day_05/files_dir/fastq4_2.fq
/data/cwbd/day_05/files_dir/fastq_1.fq
/data/cwbd/day_05/files_dir/SRR2_1.fq
/data/cwbd/day_05/files_dir/fastq.fq
/data/cwbd/day_05/files_dir/file_2.txt
/data/cwbd/day_05/files_dir/file_5.tx

### Task 4: only the TXT files

> Create a channel that includes all TXT files in the "files_dir" directory

`channel.fromPath('files_dir/*.txt')`

In [4]:
!nextflow run channels_intro.nf --step 4


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [deadly_mayer] revision: 716a17fe13

/data/cwbd/day_05/files_dir/file_3.txt
/data/cwbd/day_05/files_dir/file_4.txt
/data/cwbd/day_05/files_dir/file_1.txt
/data/cwbd/day_05/files_dir/file_2.txt
/data/cwbd/day_05/files_dir/file_5.txt


### Task 5: fastq_1.fq and fastq_2.fq

> Create a channel that includes the files "fastq_1.fq" and "fastq_2.fq" in the "files_dir" directory

`channel.fromPath('files_dir/fastq_{1,2}.fq')`: the `{1,2}` glob matches exactly these two names.

In [5]:
!nextflow run channels_intro.nf --step 5


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [scruffy_khorana] revision: 716a17fe13

/data/cwbd/day_05/files_dir/fastq_2.fq
/data/cwbd/day_05/files_dir/fastq_1.fq


### Task 6: really all files?

> go back to the time when you included all files. Are you sure that really ALL files are included? If not, how can you include them?

No. Globs skip hidden files, so Task 3 missed `.hidden_1.txt` and `.hidden_2.txt`. The `hidden: true` option of `fromPath` includes them: `channel.fromPath('files_dir/*', hidden: true)` returns 31 files instead of 29.

In [6]:
!nextflow run channels_intro.nf --step 6


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [sharp_rosalind] revision: 716a17fe13

/data/cwbd/day_05/files_dir/fastq5_2.fq
/data/cwbd/day_05/files_dir/fastq_2.fq
/data/cwbd/day_05/files_dir/SRR3_2.fq
/data/cwbd/day_05/files_dir/fastq4_1.fq
/data/cwbd/day_05/files_dir/file_3.txt
/data/cwbd/day_05/files_dir/fastq2_2.fq
/data/cwbd/day_05/files_dir/SRR1_2.fq
/data/cwbd/day_05/files_dir/file_4.txt
/data/cwbd/day_05/files_dir/fastq2_1.fq
/data/cwbd/day_05/files_dir/SRR4_2.fq
/data/cwbd/day_05/files_dir/fastq5_1.fq
/data/cwbd/day_05/files_dir/fastq1_1.fq
/data/cwbd/day_05/files_dir/SRR5_2.fq
/data/cwbd/day_05/files_dir/file_1.txt
/data/cwbd/day_05/files_dir/.hidden_2.txt
/data/cwbd/day_05/files_dir/SRR.fq
/data/cwbd/day_05/files_dir/SRR3_1.fq
/data/cwbd/day_05/files_dir/fastq3_2.fq
/data/cwbd/day_05/files_dir/.hidden_1.txt
/data/cwbd/day_05/files_dir/fastq4_2.fq
/data/cwbd/day_05/files_dir/fastq_1.fq
/data/cwbd/day_05/files_dir/SRR2_1.fq
/data/cwbd/day_05/files_dir/f

### Task 7: file pairs

> get all filepairs in the "files_dir" directory

`channel.fromFilePairs('files_dir/*_{1,2}.fq')` emits `[key, [read_1, read_2]]`, where the key is whatever `*` matched. The unpaired `SRR.fq` and `fastq.fq` drop out.

In [7]:
!nextflow run channels_intro.nf --step 7


 N E X T F L O W   ~  version 26.04.6

Launching `channels_intro.nf` [nostalgic_hawking] revision: 716a17fe13

[fastq2, [/data/cwbd/day_05/files_dir/fastq2_1.fq, /data/cwbd/day_05/files_dir/fastq2_2.fq]]
[fastq5, [/data/cwbd/day_05/files_dir/fastq5_1.fq, /data/cwbd/day_05/files_dir/fastq5_2.fq]]
[SRR3, [/data/cwbd/day_05/files_dir/SRR3_1.fq, /data/cwbd/day_05/files_dir/SRR3_2.fq]]
[fastq4, [/data/cwbd/day_05/files_dir/fastq4_1.fq, /data/cwbd/day_05/files_dir/fastq4_2.fq]]
[fastq, [/data/cwbd/day_05/files_dir/fastq_1.fq, /data/cwbd/day_05/files_dir/fastq_2.fq]]
[SRR1, [/data/cwbd/day_05/files_dir/SRR1_1.fq, /data/cwbd/day_05/files_dir/SRR1_2.fq]]
[SRR4, [/data/cwbd/day_05/files_dir/SRR4_1.fq, /data/cwbd/day_05/files_dir/SRR4_2.fq]]
[SRR5, [/data/cwbd/day_05/files_dir/SRR5_1.fq, /data/cwbd/day_05/files_dir/SRR5_2.fq]]
[fastq1, [/data/cwbd/day_05/files_dir/fastq1_1.fq, /data/cwbd/day_05/files_dir/fastq1_2.fq]]
[SRR2, [/data/cwbd/day_05/files_dir/SRR2_1.fq, /data/cwbd/day_05/files_dir/SRR

## Basic channel operators

Solutions in `basic_channel_operations.nf`. Operators take a channel and return a new one.

### Task 1: first item

> Extract the first item from the channel

`in_ch.first()`

In [8]:
!nextflow run basic_channel_operations.nf --step 1


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [infallible_lorenz] revision: be9621ab23

1


### Task 2: last item

> Extract the last item from the channel

`in_ch.last()`

In [9]:
!nextflow run basic_channel_operations.nf --step 2


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [dreamy_becquerel] revision: be9621ab23

3


### Task 3: first two items

> Use an operator to extract the first two items from the channel

`in_ch.take(2)`

In [10]:
!nextflow run basic_channel_operations.nf --step 3


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [exotic_sanger] revision: be9621ab23

1
2


### Task 4: squared values

> Return the squared values of the channel

`in_ch.map { v -> v * v }`

In [11]:
!nextflow run basic_channel_operations.nf --step 4


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [hungry_fermi] revision: be9621ab23

4
9
16


### Task 5: first two squared values

> Remember the previous task where you squared the values of the channel. Now, extract the first two items from the squared channel

Already solved in the template by chaining both operators: `in_ch.map { it -> it * it }.take(2)`.

In [12]:
!nextflow run basic_channel_operations.nf --step 5


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [insane_pike] revision: be9621ab23

4
9


### Task 6: reverse with map

> Remember when you used bash to reverse the output? Try to use map and Groovy to reverse the output

`in_ch.map { s -> s.reverse() }`: each string is reversed by the Groovy string method inside `map`.

In [13]:
!nextflow run basic_channel_operations.nf --step 6


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [loving_gauss] revision: be9621ab23

rolyaT
tfiwS


### Task 7: file name and path

> Use fromPath to include all fastq files in the "files_dir" directory, then use map to return a pair containing the file name and the file path (Hint: include groovy code)

`in_ch.map { f -> [f.name, f] }`: every item is a Path, and `.name` is its file name.

In [14]:
!nextflow run basic_channel_operations.nf --step 7


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [infallible_meucci] revision: be9621ab23

[fastq5_2.fq, /data/cwbd/day_05/files_dir/fastq5_2.fq]
[fastq_2.fq, /data/cwbd/day_05/files_dir/fastq_2.fq]
[SRR3_2.fq, /data/cwbd/day_05/files_dir/SRR3_2.fq]
[fastq4_1.fq, /data/cwbd/day_05/files_dir/fastq4_1.fq]
[fastq2_2.fq, /data/cwbd/day_05/files_dir/fastq2_2.fq]
[SRR1_2.fq, /data/cwbd/day_05/files_dir/SRR1_2.fq]
[fastq2_1.fq, /data/cwbd/day_05/files_dir/fastq2_1.fq]
[SRR4_2.fq, /data/cwbd/day_05/files_dir/SRR4_2.fq]
[fastq5_1.fq, /data/cwbd/day_05/files_dir/fastq5_1.fq]
[fastq1_1.fq, /data/cwbd/day_05/files_dir/fastq1_1.fq]
[SRR5_2.fq, /data/cwbd/day_05/files_dir/SRR5_2.fq]
[SRR.fq, /data/cwbd/day_05/files_dir/SRR.fq]
[SRR3_1.fq, /data/cwbd/day_05/files_dir/SRR3_1.fq]
[fastq3_2.fq, /data/cwbd/day_05/files_dir/fastq3_2.fq]
[fastq4_2.fq, /data/cwbd/day_05/files_dir/fastq4_2.fq]
[fastq_1.fq, /data/cwbd/day_05/files_dir/fastq_1.fq]
[SRR2_1.fq, /data/cwbd/day_05/fi

### Task 8: combine two channels

> Combine the items from the two channels into a single channel

`ch_1.mix(ch_2)` merges both channels into one. The items come out in whatever order they arrive, so 1 to 6 are interleaved. (The `combine` operator would instead give every pair `[1, 4]`, `[1, 5]`, ...)

In [15]:
!nextflow run basic_channel_operations.nf --step 8


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [wise_swirles] revision: be9621ab23

1
4
5
6
2
3


### Task 9: flatten

> Flatten the list in the channel

`in_ch.flatten()` turns the two lists into six single items.

In [16]:
!nextflow run basic_channel_operations.nf --step 9


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [special_pike] revision: be9621ab23

1
2
3
4
5
6


### Task 10: collect

> Collect the items of a channel into a list. What kind of channel is the output channel?

`in_ch.collect()` emits one list `[1, 2, 3]`.

In [17]:
!nextflow run basic_channel_operations.nf --step 10


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [determined_becquerel] revision: be9621ab23

[1, 2, 3]


What kind of channel is the output channel?

A value channel. `collect` waits until the input is complete and then emits a single item, which, unlike an item of a queue channel, can be read by any number of processes or operators without being used up.

### Task 11: group by the first element

> From the input channel, create lists where each first item in the list of lists is the first item in the output channel, followed by a list of all the items its paired with

`in_ch.groupTuple()` groups by the first element and collects all second elements of that key in arrival order.

In [18]:
!nextflow run basic_channel_operations.nf --step 11


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [shrivelled_bartik] revision: be9621ab23

[1, [V, f, B]]
[3, [M, G, 33]]
[2, [O, L, E]]


### Task 12: join

> Create a channel that joins the input to the output channel. What do you notice?

`left_ch.join(right_ch)` matches items of the two channels by their first element.

In [19]:
!nextflow run basic_channel_operations.nf --step 12


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [spontaneous_thompson] revision: be9621ab23

[1, V, f]
[3, M, G]
[2, O, L]


Task 12 - What do you notice compared to Task 11?

`groupTuple` collected *every* value of a key. `join` pairs items one to one: the first left item with key 1 meets the first right item with key 1, and so on. Every key appears only once in the output, and the leftovers without a partner (`B` and `33` from the left, `E` from the right) are silently dropped. The option `remainder: true` would emit them as well.

### Task 13: even and odd

> Split the input channel into two channels, one of all the even numbers and the other of all the odd numbers. \
Write them to stdout including information about which is which

`branch` routes each number to the first label whose condition holds (`even: v % 2 == 0`, `odd: true`). Each branch is collected into a list and printed with `dump(tag: 'even')` / `dump(tag: 'odd')`, which only prints when the run has `-dump-channels` and labels every line with its tag.

In [20]:
!nextflow run basic_channel_operations.nf --step 13 -dump-channels


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [special_lorenz] revision: be9621ab23

[DUMP: odd] [1, 3, 5, 7, 9]
[DUMP: even] [2, 4, 6, 8, 10]


### Task 14: names to a file

> Nextflow has the concept of maps. Write the names in the maps in this channel to a file called "names.txt". Each name should be on a new line. \
Store the file in the "results" directory under the name "names.txt"

`map { person -> person.name }` picks the name out of each map, and `collectFile(name: 'names.txt', newLine: true, sort: 'index', storeDir: 'results')` writes them one per line, in channel order, to `results/names.txt`.

In [21]:
!nextflow run basic_channel_operations.nf --step 14

!cat results/names.txt


 N E X T F L O W   ~  version 26.04.6

Launching `basic_channel_operations.nf` [cheesy_varahamihira] revision: be9621ab23

Harry
Ron
Hermione
Albus
Snape
Hagrid
Dobby


## Processes

Solutions in `basics_processes.nf`.

### Task 1: Hello World

> create a process that says Hello World! (add debug true to the process right after initializing to be sable to print the output to the console)

`SAYHELLO` runs `echo "Hello World!"`; `debug true` forwards the task's stdout to the console.

In [22]:
!nextflow run basics_processes.nf --step 1


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [trusting_curry] revision: 877ba56f01

[-        ] SAYHELLO | 0 of 1

executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/85/a7e0e635cf75b575d2062855471a0485/a7e0e6]8;;] SAYHELLO | 1 of 1 ✔
Hello World!




### Task 2: Hello World in Python

> create a process that says Hello World! using Python

The script block starts with the shebang `#!/usr/bin/env python3`, so the task runs it with Python instead of Bash.

In [23]:
!nextflow run basics_processes.nf --step 2


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [compassionate_minsky] revision: 877ba56f01

executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/e9/98b744279988bfa83f9df726da87f7e9/98b744]8;;] SAYHELLO_PYTHON | 0 of 1
Hello World!


executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/e9/98b744279988bfa83f9df726da87f7e9/98b744]8;;] SAYHELLO_PYTHON | 1 of 1 ✔
Hello World!




### Task 3: string from a channel to the command line

> create a process that reads in the string "Hello world!" from a channel and write it to command line

`SAYHELLO_PARAM` takes `val greeting` as input and echoes it.

In [24]:
!nextflow run basics_processes.nf --step 3


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [curious_hamilton] revision: 877ba56f01

[-        ] SAYHELLO_PARAM | 0 of 1

executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/ac/352aeb4bc6ed5b4a8301bbdcea6f69ac/352aeb]8;;] SAYHELLO_PARAM (1) | 1 of 1 ✔
Hello world!




### Task 4: string from a channel to a file

> create a process that reads in the string "Hello world!" from a channel and write it to a file.

`SAYHELLO_FILE` writes `greeting.txt` and declares it as `path 'greeting.txt'` output. Nothing is printed this time.

In [25]:
!nextflow run basics_processes.nf --step 4


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [astonishing_franklin] revision: 877ba56f01

[-        ] SAYHELLO_FILE | 0 of 1

executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/ab/04ab9ffb3631140fb3630ca9b442a7ab/04ab9f]8;;] SAYHELLO_FILE (1) | 1 of 1 ✔



> It might be that you created the file but it doesnt appear in the directory. Use the work folder and the output on the command line to find the file.

The file stays in the task's own work directory. The hash in the log line, here `[c6/2f1722]`, is the start of that directory's path, `work/c6/2f1722.../greeting.txt`. Without `publishDir` or a workflow output, nothing is copied to the launch directory.

In [26]:
!cat work/*/*/greeting.txt

Hello world!
Hello world!


### Task 5: uppercase into a file

> create a process that reads in a string and converts it to uppercase and saves it to a file as output. View the path to the file in the console

`UPPERCASE` pipes the string through `tr '[:lower:]' '[:upper:]'` into `upper.txt`; `out_ch.view()` prints the path of that output file.

In [27]:
!nextflow run basics_processes.nf --step 5


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [reverent_franklin] revision: 877ba56f01

[-        ] UPPERCASE | 0 of 1

executor >  local (1)
[]8;;file:///data/cwbd/day_05/work/13/ef7cb3ce613172ce8c19e26000c80713/ef7cb3]8;;] UPPERCASE (1) | 1 of 1 ✔
/data/cwbd/day_05/work/13/ef7cb3ce613172ce8c19e26000c807/upper.txt



### Task 6: print the uppercase file

> add another process that reads in the resulting file from UPPERCASE and print the content to the console (debug true).

`PRINTUPPER` takes `path upper_file` as input and runs `cat` on it.

In [28]:
!nextflow run basics_processes.nf --step 6


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [thirsty_morse] revision: 877ba56f01

[-        ] UPPERCASE  | 0 of 1
[-        ] PRINTUPPER -

executor >  local (2)
[]8;;file:///data/cwbd/day_05/work/61/16d529477498024a33f6c6ef1b1a6761/16d529]8;;] UPPERCASE (1)  | 1 of 1 ✔
[]8;;file:///data/cwbd/day_05/work/4c/c2598b63a4854eb0f620420406fac04c/c2598b]8;;] PRINTUPPER (1) | 1 of 1 ✔
HELLO WORLD!




Comared to all the other runs. What changed in the output here and why?

For the first time two processes run in one workflow, so there are two task lines with two different hashes, and `PRINTUPPER` is only submitted after `UPPERCASE` has finished: its input is the output of the first. Nextflow does not move the file, it links `upper.txt` from the `UPPERCASE` work directory into the new task's directory. And the console now shows the file's *content* (`HELLO WORLD!`), printed by the task itself via `debug true`, instead of the path that `view()` showed in Task 5.

### Task 7: zip in one format

> based on the paramater "zip" (see at the head of the file), create a process that zips the file created in the UPPERCASE process either in "zip", "gzip" OR "bzip2" format. \
Print out the path to the zipped file in the console

`ZIPFILE` takes `tuple path(in_file), val(format)` and picks its script with an `if` on the format; an unknown format stops the run with an error. The workflow pairs the file with `params.zip` via `combine`. The default is `zip`; `--zip gzip` or `--zip bzip2` switch it.

In [29]:
!nextflow run basics_processes.nf --step 7


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [naughty_magritte] revision: 877ba56f01

[-        ] UPPERCASE -
[-        ] ZIPFILE   -

executor >  local (2)
[]8;;file:///data/cwbd/day_05/work/27/70bbf5d4a7ed8a9ab2c33e49d1491527/70bbf5]8;;] UPPERCASE (1) | 1 of 1 ✔
[]8;;file:///data/cwbd/day_05/work/d0/8a878a8bc2e9dd84454887c847603cd0/8a878a]8;;] ZIPFILE (1)   | 1 of 1 ✔
/data/cwbd/day_05/work/d0/8a878a8bc2e9dd84454887c847603c/upper.txt.zip

executor >  local (2)
[]8;;file:///data/cwbd/day_05/work/27/70bbf5d4a7ed8a9ab2c33e49d1491527/70bbf5]8;;] UPPERCASE (1) | 1 of 1 ✔
[]8;;file:///data/cwbd/day_05/work/d0/8a878a8bc2e9dd84454887c847603cd0/8a878a]8;;] ZIPFILE (1)   | 1 of 1 ✔
/data/cwbd/day_05/work/d0/8a878a8bc2e9dd84454887c847603c/upper.txt.zip



### Task 8: zip in all three formats

> Create a process that zips the file created in the UPPERCASE process in "zip", "gzip" AND "bzip2" format. Print out the paths to the zipped files in the console

Same process, but `combine` with `channel.of('zip', 'gzip', 'bzip2')` gives three items, so `ZIPFILE` runs as three parallel tasks.

In [30]:
!nextflow run basics_processes.nf --step 8


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [trusting_cajal] revision: 877ba56f01

[-        ] UPPERCASE | 0 of 1
[-        ] ZIPFILE   -

executor >  local (4)
[]8;;file:///data/cwbd/day_05/work/30/8aec8e3797e1ea46044fd0dc18a70830/8aec8e]8;;] UPPERCASE (1) | 1 of 1 ✔
[]8;;file:///data/cwbd/day_05/work/d5/96fee40d67f84d4d31ea4409ce50a0d5/96fee4]8;;] ZIPFILE (1)   | 3 of 3 ✔
/data/cwbd/day_05/work/43/f295e518fbc52920d702224da9fb4a/upper.txt.bz2
/data/cwbd/day_05/work/f5/3e454106ae37b2ae4a68c5dff98384/upper.txt.gz
/data/cwbd/day_05/work/d5/96fee40d67f84d4d31ea4409ce50a0/upper.txt.zip



### Task 9: names and titles to a TSV

> Create a process that reads in a list of names and titles from a channel and writes them to a file. \
Store the file in the "results" directory under the name "names.tsv"

`WRITETOFILE` writes one tab separated row per map, then `collectFile(name: 'names.tsv', seed: "name\ttitle\n", storeDir: 'results')` merges the rows below a header line. The template's `in_ch | WRITETOFILE` is written as `WRITETOFILE(in_ch).collectFile(...)`, because the strict parser cannot pipe into an operator with arguments. The tasks run in parallel, so the row order changes between runs.

In [31]:
!nextflow run basics_processes.nf --step 9


 N E X T F L O W   ~  version 26.04.6

Launching `basics_processes.nf` [silly_raman] revision: 877ba56f01

[-        ] WRITETOFILE -

executor >  local (7)
[]8;;file:///data/cwbd/day_05/work/32/ae02998f7cc2b1192a5ae9a31a9bbd32/ae0299]8;;] WRITETOFILE (4) | 7 of 7 ✔

executor >  local (7)
[]8;;file:///data/cwbd/day_05/work/32/ae02998f7cc2b1192a5ae9a31a9bbd32/ae0299]8;;] WRITETOFILE (4) | 7 of 7 ✔



In [32]:
import pandas as pd

df = pd.read_csv("results/names.tsv", sep="\t")
df

,name,title
0,Albus,headmaster
1,Ron,student
2,Dobby,hero
3,Snape,teacher
4,Hermione,student
5,Harry,student
6,Hagrid,groundkeeper


## Advanced channel operations: meta maps

Solutions in `advanced_channel_operations.nf`. A meta map carries the sample's metadata next to its files through the whole pipeline, as in nf-core.

### Task 1: read the samplesheet

> Read in the samplesheet.

`channel.fromPath('samplesheet.csv').splitCsv(header: true)` emits one map per row, keyed by the column names.

In [33]:
!nextflow run advanced_channel_operations.nf --step 1


 N E X T F L O W   ~  version 26.04.6

Launching `advanced_channel_operations.nf` [hopeful_blackwell] revision: 0fc5b01cf5

[sample:CONTROL_REP1, fastq_1:fq_1_R1.fastq.gz, fastq_2:fq_1_R2.fastq.gz, strandedness:auto]
[sample:CONTROL_REP2, fastq_1:fq_2_R1.fastq.gz, fastq_2:fq_2_R2.fastq.gz, strandedness:forward]
[sample:CONTROL_REP3, fastq_1:fq_3_R1.fastq.gz, fastq_2:fq_3_R2.fastq.gz, strandedness:reverse]
[sample:CONTROL_REP1, fastq_1:fq_4_R1.fastq.gz, fastq_2:fq_4_R2.fastq.gz, strandedness:auto]


### Task 2: meta map plus reads

> Read in the samplesheet and create a meta-map with all metadata and another list with the filenames ([[metadata_1 : metadata_1, ...], [fastq_1, fastq_2]]). \
Set the output to a new channel "in_ch" and view the channel. YOU WILL NEED TO COPY AND PASTE THIS CODE INTO SOME OF THE FOLLOWING TASKS (sorry for that).

`.map { row -> [[sample: row.sample, strandedness: row.strandedness], [row.fastq_1, row.fastq_2]] }`

In [34]:
!nextflow run advanced_channel_operations.nf --step 2


 N E X T F L O W   ~  version 26.04.6

Launching `advanced_channel_operations.nf` [suspicious_wing] revision: 0fc5b01cf5

[[sample:CONTROL_REP1, strandedness:auto], [fq_1_R1.fastq.gz, fq_1_R2.fastq.gz]]
[[sample:CONTROL_REP2, strandedness:forward], [fq_2_R1.fastq.gz, fq_2_R2.fastq.gz]]
[[sample:CONTROL_REP3, strandedness:reverse], [fq_3_R1.fastq.gz, fq_3_R2.fastq.gz]]
[[sample:CONTROL_REP1, strandedness:auto], [fq_4_R1.fastq.gz, fq_4_R2.fastq.gz]]


### Task 3: split by strandedness

> Now we assume that we want to handle different "strandedness" values differently. \
Split the channel into the right amount of channels and write them all to stdout so that we can understand which is which.

The samplesheet has three values, so `branch` gets three labels (`auto`, `forward`, `reverse`) tested on `meta.strandedness`. Each branch is printed with its own `dump` tag.

In [35]:
!nextflow run advanced_channel_operations.nf --step 3 -dump-channels


 N E X T F L O W   ~  version 26.04.6

Launching `advanced_channel_operations.nf` [desperate_torvalds] revision: 0fc5b01cf5

[DUMP: auto] [['sample':'CONTROL_REP1', 'strandedness':'auto'], ['fq_1_R1.fastq.gz', 'fq_1_R2.fastq.gz']]
[DUMP: forward] [['sample':'CONTROL_REP2', 'strandedness':'forward'], ['fq_2_R1.fastq.gz', 'fq_2_R2.fastq.gz']]
[DUMP: reverse] [['sample':'CONTROL_REP3', 'strandedness':'reverse'], ['fq_3_R1.fastq.gz', 'fq_3_R2.fastq.gz']]
[DUMP: auto] [['sample':'CONTROL_REP1', 'strandedness':'auto'], ['fq_4_R1.fastq.gz', 'fq_4_R2.fastq.gz']]


### Task 4: group by sample and strandedness

> Group together all files with the same sample-id and strandedness value.

Both values are already in the meta map, so `groupTuple()` can use the whole map as its key. The two `CONTROL_REP1` / `auto` rows become one item with two read pairs.

In [36]:
!nextflow run advanced_channel_operations.nf --step 4


 N E X T F L O W   ~  version 26.04.6

Launching `advanced_channel_operations.nf` [focused_murdock] revision: 0fc5b01cf5

[[sample:CONTROL_REP1, strandedness:auto], [[fq_1_R1.fastq.gz, fq_1_R2.fastq.gz], [fq_4_R1.fastq.gz, fq_4_R2.fastq.gz]]]
[[sample:CONTROL_REP2, strandedness:forward], [[fq_2_R1.fastq.gz, fq_2_R2.fastq.gz]]]
[[sample:CONTROL_REP3, strandedness:reverse], [[fq_3_R1.fastq.gz, fq_3_R2.fastq.gz]]]


## Linking processes and channels

Solutions in `link_p_c.nf`: the samplesheet `samplesheet_2.csv` is read with `splitCsv` and mapped to `[[block_size: ...], input_str, out_name]`. `SPLITLETTERS` runs `split -b <block_size>` on the string and emits `[meta, chunk files]`, the files named `<out_name>_aa`, `_ab`, .... A `map` drops the meta map and `flatten` turns the file lists into single files. `CONVERTTOUPPER` uppercases each chunk, prints it (`debug true`) and copies the result to `results/` via `publishDir`.

In [37]:
!nextflow run link_p_c.nf


 N E X T F L O W   ~  version 26.04.6

Launching `link_p_c.nf` [ecstatic_euler] revision: a2f71e04e0

[-        ] SPLITLETTERS   | 0 of 2
[-        ] CONVERTTOUPPER -

executor >  local (13)
[]8;;file:///data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0de7/4a2cec]8;;] SPLITLETTERS (2)   | 2 of 2 ✔
[]8;;file:///data/cwbd/day_05/work/d5/8eaba087ac6ccca67eaa8887d67a02d5/8eaba0]8;;] CONVERTTOUPPER (4) | 11 of 11 ✔
O WO

RLD

ATI

ORK

WS

HELL

COM

PUT

L W

FLO

ONA


executor >  local (13)
[]8;;file:///data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0de7/4a2cec]8;;] SPLITLETTERS (2)   | 2 of 2 ✔
[]8;;file:///data/cwbd/day_05/work/d5/8eaba087ac6ccca67eaa8887d67a02d5/8eaba0]8;;] CONVERTTOUPPER (4) | 11 of 11 ✔
O WO

RLD

ATI

ORK

WS

HELL

COM

PUT

L W

FLO

ONA




### Give a list with the paths to the chunk files

> Give a list with the paths to the chunk files

The chunks sit in the work directories of the two `SPLITLETTERS` tasks; `nextflow log` returns those directories for the last run.

In [38]:
from pathlib import Path

split_dirs = !nextflow log last -f workdir -F 'process == "SPLITLETTERS"'
chunk_paths = sorted(str(p) for d in split_dirs for p in Path(d).glob("*_a?"))
chunk_paths

['/data/cwbd/day_05/work/3e/7b94bbdbc58e3f147c4d6c6763c9bd/h_w_aa',
 '/data/cwbd/day_05/work/3e/7b94bbdbc58e3f147c4d6c6763c9bd/h_w_ab',
 '/data/cwbd/day_05/work/3e/7b94bbdbc58e3f147c4d6c6763c9bd/h_w_ac',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_aa',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ab',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ac',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ad',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ae',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_af',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ag',
 '/data/cwbd/day_05/work/e7/4a2ceca25574f5565858da631e6b0d/c_w_ah']

### Why was CONVERTTOUPPER run so often?

> Why was CONVERTTOUPPER run so often?

A process runs one task per item on its input channel. After `flatten` every chunk file is its own item: "Hello World" in blocks of 4 gives 3 chunks and "Computational Workflows" in blocks of 3 gives 8, so `CONVERTTOUPPER` ran 11 times while `SPLITLETTERS` ran only twice, once per samplesheet row. The 11 tasks are independent, so they also finish in no fixed order, which is why the printed pieces are shuffled.